# Utility Usage Prediction Tool

## Model Training Notebook

### CodeVedX AI/ML Internship

This notebook performs:

- Loading the processed dataset
- Dataset exploration and analysis
- Feature selection and engineering
- Train-test split
- Linear Regression model training
- Model evaluation with metrics
- Model saving with Joblib
- Visualization generation

**Target Variable:** Global_active_power (kilowatts)

**Model:** Linear Regression

**Framework:** Scikit-Learn

In [ ]:
# ========================================
# STEP 1: Import Required Libraries
# ========================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ All libraries imported successfully")

In [ ]:
# ========================================
# STEP 2: Setup Paths
# ========================================

PROJECT_ROOT = Path("..")

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "utility_usage.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "utility_usage_model.pkl"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CHARTS_DIR = OUTPUTS_DIR / "charts"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
REPORTS_DIR = OUTPUTS_DIR / "reports"

# Create output directories if they don't exist
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MODEL_PATH.parent, exist_ok=True)

print(f"Processed Data: {PROCESSED_DATA}")
print(f"Model Path: {MODEL_PATH}")
print(f"Charts Directory: {CHARTS_DIR}")
print(f"Predictions Directory: {PREDICTIONS_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

In [ ]:
# ========================================
# STEP 3: Load Processed Dataset
# ========================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = pd.read_csv(PROCESSED_DATA)

print(f"✓ Dataset loaded successfully")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Memory Usage: {df.memory_usage().sum() / 1024**2:.2f} MB")

In [ ]:
# ========================================
# STEP 4: Dataset Exploration
# ========================================

print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)

# Display first few rows
print("\n1. First 5 rows:")
display(df.head())

# Display dataset info
print("\n2. Dataset Info:")
print(df.info())

# Display statistical summary
print("\n3. Statistical Summary:")
display(df.describe())

# Check for missing values
print("\n4. Missing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✓ No missing values found")

# Check for duplicates
print(f"\n5. Duplicate Rows: {df.duplicated().sum()}")

# Display column names
print(f"\n6. Columns ({len(df.columns)}):")
for col in df.columns:
    print(f"   - {col}")

In [ ]:
# ========================================
# STEP 5: Feature Selection
# ========================================

print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)

# Define target variable
TARGET_COLUMN = "Global_active_power"

# Select features for training
# We'll use numeric features that are available at prediction time
FEATURE_COLUMNS = [
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "Year",
    "Month",
    "Day",
    "Hour"
]

# Prepare features (X) and target (y)
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print(f"\n✓ Features selected: {len(FEATURE_COLUMNS)}")
print(f"  Target variable: {TARGET_COLUMN}")
print(f"\nFeature columns:")
for i, col in enumerate(FEATURE_COLUMNS, 1):
    print(f"  {i}. {col}")

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# ========================================
# STEP 6: Train-Test Split
# ========================================

print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)

# Split data: 80% training, 20% testing
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"\n✓ Data split completed")
print(f"  Training set: {X_train.shape[0]} samples ({100*(1-TEST_SIZE):.0f}%)")
print(f"  Testing set: {X_test.shape[0]} samples ({100*TEST_SIZE:.0f}%)")
print(f"  Random State: {RANDOM_STATE}")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Testing target shape: {y_test.shape}")

In [ ]:
# ========================================
# STEP 7: Linear Regression Model Training
# ========================================

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

# Initialize the model
model = LinearRegression()

# Train the model
print("\nTraining Linear Regression model...")
model.fit(X_train, y_train)

print("✓ Model trained successfully")
print(f"\nModel Parameters:")
print(f"  Coefficients: {model.coef_}")
print(f"  Intercept: {model.intercept_:.4f}")

In [ ]:
# ========================================
# STEP 8: Make Predictions
# ========================================

print("=" * 60)
print("MAKING PREDICTIONS")
print("=" * 60)

# Predict on training data
y_train_pred = model.predict(X_train)

# Predict on testing data
y_test_pred = model.predict(X_test)

print("✓ Predictions completed")
print(f"  Training predictions: {len(y_train_pred)} samples")
print(f"  Testing predictions: {len(y_test_pred)} samples")
print(f"\nSample predictions (first 5):")
for i in range(5):
    actual = y_test.iloc[i]
    predicted = y_test_pred[i]
    error = abs(actual - predicted)
    print(f"  {i+1}. Actual: {actual:.4f} kW | Predicted: {predicted:.4f} kW | Error: {error:.4f} kW")

In [ ]:
# ========================================
# STEP 9: Model Evaluation
# ========================================

print("=" * 60)
print("MODEL EVALUATION METRICS")
print("=" * 60)

# Calculate metrics for training set
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_train_pred)

# Calculate metrics for testing set
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

# Display metrics
print("\n📊 Training Set Metrics:")
print(f"  MAE  (Mean Absolute Error):      {train_mae:.4f} kW")
print(f"  MSE  (Mean Squared Error):       {train_mse:.4f}")
print(f"  RMSE (Root Mean Squared Error):  {train_rmse:.4f} kW")
print(f"  R² Score:                        {train_r2:.4f}")

print("\n📊 Testing Set Metrics:")
print(f"  MAE  (Mean Absolute Error):      {test_mae:.4f} kW")
print(f"  MSE  (Mean Squared Error):       {test_mse:.4f}")
print(f"  RMSE (Root Mean Squared Error):  {test_rmse:.4f} kW")
print(f"  R² Score:                        {test_r2:.4f}")

# Store metrics for report
metrics = {
    "train_mae": train_mae,
    "train_mse": train_mse,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_mae": test_mae,
    "test_mse": test_mse,
    "test_rmse": test_rmse,
    "test_r2": test_r2
}

In [ ]:
# ========================================
# STEP 10: Save Model
# ========================================

print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

# Save model using Joblib
joblib.dump(model, MODEL_PATH)

print(f"✓ Model saved successfully")
print(f"  Path: {MODEL_PATH}")
print(f"  File size: {os.path.getsize(MODEL_PATH) / 1024**2:.2f} MB")

In [ ]:
# ========================================
# STEP 11: Generate Visualization Charts
# ========================================

print("=" * 60)
print("GENERATING CHARTS")
print("=" * 60)

# Chart 1: Histogram of Global_active_power
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Global_active_power'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Global Active Power (kW)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Global Active Power', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'histogram_global_active_power.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Chart 1: Histogram saved")

# Chart 2: Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Chart 2: Correlation Heatmap saved")

# Chart 3: Actual vs Predicted
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(y_test, y_test_pred, alpha=0.5, color='steelblue', edgecolors='black', linewidth=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Global Active Power (kW)', fontsize=12)
ax.set_ylabel('Predicted Global Active Power (kW)', fontsize=12)
ax.set_title('Actual vs Predicted Values', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Chart 3: Actual vs Predicted saved")

# Chart 4: Residual Plot
residuals = y_test - y_test_pred

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(y_test_pred, residuals, alpha=0.5, color='steelblue', edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='r', linestyle='--', lw=2, label='Zero Error')
ax.set_xlabel('Predicted Values (kW)', fontsize=12)
ax.set_ylabel('Residuals (kW)', fontsize=12)
ax.set_title('Residual Plot', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'residual_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Chart 4: Residual Plot saved")

# Chart 5: Feature Distribution (Box Plots)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, feature in enumerate(FEATURE_COLUMNS):
    axes[idx].boxplot(df[feature], vert=True)
    axes[idx].set_title(f'{feature}', fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Value')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Feature Distribution (Box Plots)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'feature_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Chart 5: Feature Distribution saved")

print(f"\n✓ All charts saved to: {CHARTS_DIR}")

In [ ]:
# ========================================
# STEP 12: Generate Prediction CSV
# ========================================

print("=" * 60)
print("GENERATING PREDICTION REPORT")
print("=" * 60)

# Create predictions dataframe
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred,
    'Error': y_test.values - y_test_pred,
    'Absolute_Error': np.abs(y_test.values - y_test_pred)
})

# Add feature columns
for col in FEATURE_COLUMNS:
    predictions_df[col] = X_test[col].values

# Reorder columns
column_order = FEATURE_COLUMNS + ['Actual', 'Predicted', 'Error', 'Absolute_Error']
predictions_df = predictions_df[column_order]

# Save to CSV
predictions_path = PREDICTIONS_DIR / 'model_predictions.csv'
predictions_df.to_csv(predictions_path, index=False)

print(f"✓ Predictions saved successfully")
print(f"  Path: {predictions_path}")
print(f"  Records: {len(predictions_df)}")
print(f"\nFirst 10 predictions:")
display(predictions_df.head(10))

In [ ]:
# ========================================
# STEP 13: Generate Model Evaluation Report
# ========================================

print("=" * 60)
print("GENERATING EVALUATION REPORT")
print("=" * 60)

# Create comprehensive report
report = f"""
================================================================================
                    MODEL EVALUATION REPORT
================================================================================

Model Information:
  - Model Type: Linear Regression
  - Framework: Scikit-Learn
  - Target Variable: Global_active_power (kW)
  - Features Used: {len(FEATURE_COLUMNS)}

Dataset Information:
  - Total Samples: {df.shape[0]}
  - Training Samples: {X_train.shape[0]} ({(1-TEST_SIZE)*100:.0f}%)
  - Testing Samples: {X_test.shape[0]} ({TEST_SIZE*100:.0f}%)
  - Features: {len(FEATURE_COLUMNS)}

Model Performance Metrics:
  Training Set:
    - Mean Absolute Error (MAE):  {train_mae:.4f} kW
    - Mean Squared Error (MSE):   {train_mse:.4f}
    - Root Mean Squared Error:    {train_rmse:.4f} kW
    - R² Score:                   {train_r2:.4f}

  Testing Set:
    - Mean Absolute Error (MAE):  {test_mae:.4f} kW
    - Mean Squared Error (MSE):   {test_mse:.4f}
    - Root Mean Squared Error:    {test_rmse:.4f} kW
    - R² Score:                   {test_r2:.4f}

Model Parameters:
  - Coefficients: {model.coef_}
  - Intercept: {model.intercept_:.4f}

Feature Importance (Coefficients):
  """

# Add feature importance
for feature, coef in zip(FEATURE_COLUMNS, model.coef_):
    report += f"    - {feature:25s}: {coef:+.6f}\n"

report += """

Outputs Generated:
  - Model: {MODEL_PATH.name}
  - Charts: {CHARTS_DIR}
    - histogram_global_active_power.png
    - correlation_heatmap.png
    - actual_vs_predicted.png
    - residual_plot.png
    - feature_distribution.png
  - Predictions: {predictions_path.name}

================================================================================
                              END OF REPORT
================================================================================
"""

# Save report
report_path = REPORTS_DIR / 'model_evaluation_report.txt'
with open(report_path, 'w') as f:
    f.write(report)

print(report)
print(f"\n✓ Report saved to: {report_path}")

## ✅ Model Training Completed Successfully!

### Summary:

- **Model:** Linear Regression
- **Features:** 10 features selected
- **Training Samples:** 80% of dataset
- **Testing Samples:** 20% of dataset
- **R² Score:** Model performance on test set

### Generated Outputs:

1. **Model File:** `models/utility_usage_model.pkl`
2. **Charts:** `outputs/charts/` (5 visualizations)
3. **Predictions:** `outputs/predictions/model_predictions.csv`
4. **Report:** `outputs/reports/model_evaluation_report.txt`

### Next Steps:

The trained model can now be used for:
- Predicting utility usage based on input features
- Analyzing feature importance
- Making real-time predictions through the console application